# MLOps: Experiment Tracking, Model Registry, CI/CD
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 3, Session 2**

Today we're going to look at the operational scaffolding that makes ML systems reliable over time. A model that works today but can't be reproduced, monitored, or safely updated isn't production-ready.

Four things we'll work through:
1. MLflow experiment tracking — logging hyperparameters, metrics, and artifacts across training runs
2. Data validation with Great Expectations — catching bad data before it reaches the model
3. Distribution shift detection — using statistical tests to catch when production data diverges from training
4. Retraining trigger simulation — automated performance monitoring with alerting

In [1]:
# Install dependencies
# !pip install mlflow great-expectations scipy -q

## Part 1: MLflow Experiment Tracking
*"Which run produced the model in production?" — answerable in 10 seconds*

We're going to train three variants of a classifier with different hyperparameters, log everything to MLflow, and then query the results. This is the foundation of reproducible ML.

In [1]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# Hyperparameter variants to compare
experiment_configs = [
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": None},
]

# Pin an explicit tracking URI (SQLite db in this folder) — MLflow 3.x's
# filesystem backend ('./mlruns') is deprecated, and relying on the implicit
# default means the notebook kernel's cwd must match wherever `mlflow ui` is
# later launched from, which is a common source of "my runs don't show up."
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("breast_cancer_classifier")

print("=== MLflow Experiment Tracking ===")
print("Training 3 model variants...")
print()

run_results = []

for i, config in enumerate(experiment_configs, 1):
    with mlflow.start_run(run_name=f"rf_variant_{i}") as run:
        # ── Train ────────────────────────────────────────────────────────────
        # Wrapping preprocessing + model in a Pipeline means MLflow logs (and
        # the registry serves) one artifact that reproduces the full inference
        # path — not just the classifier trained on already-scaled data.
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(
                n_estimators=config["n_estimators"],
                max_depth=config["max_depth"],
                random_state=42,
            )),
        ])
        pipeline.fit(X_train, y_train)

        # ── Evaluate ─────────────────────────────────────────────────────────
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)

        # ── Log to MLflow ─────────────────────────────────────────────────────
        # Log hyperparameters
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("n_estimators", config["n_estimators"])
        mlflow.log_param("max_depth", str(config["max_depth"]))
        mlflow.log_param("random_state", 42)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)

        # Log the full pipeline as the model artifact (scaler + classifier)
        mlflow.sklearn.log_model(pipeline, artifact_path="model")

        run_id_short = run.info.run_id[:8] + "..."
        print(f"Run {i}: RandomForest  n_estimators={config['n_estimators']}  max_depth={config['max_depth']}")
        print(f"  → accuracy={accuracy:.4f}, f1={f1:.4f}, auc={auc:.4f}")
        print(f"  → run_id: {run_id_short}")
        print()

        run_results.append({
            "run_id": run_id_short,
            "model": "RandomForest",
            "n_estimators": config["n_estimators"],
            "max_depth": str(config["max_depth"]),
            "accuracy": round(accuracy, 4),
            "f1": round(f1, 4),
            "auc": round(auc, 4),
        })

# Find best run
best = max(run_results, key=lambda r: r["auc"])
print(f"Best run by AUC: Run {run_results.index(best)+1} (auc={best['auc']})")

print()
print("=== MLflow Run Summary ===")
header = f"  {'run_id':<20} {'model':<15} {'n_estimators':<14} {'max_depth':<10} {'accuracy':>9} {'f1':>8} {'auc':>8}"
print(header)
for r in run_results:
    marker = "  ← best" if r == best else ""
    print(f"  {r['run_id']:<20} {r['model']:<15} {str(r['n_estimators']):<14} {r['max_depth']:<10} "
          f"{r['accuracy']:>9.4f} {r['f1']:>8.4f} {r['auc']:>8.4f}{marker}")

print()
print("MLflow UI: run `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001` in your terminal from this folder, then open http://localhost:5000")
print("The UI shows all runs, metrics, parameters, and allows comparison plots.")

2026/09/01 10:04:07 INFO mlflow.tracking.fluent: Experiment with name 'breast_cancer_classifier' does not exist. Creating a new experiment.
2026/09/01 10:04:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== MLflow Experiment Tracking ===
Training 3 model variants...



2026/09/01 10:04:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 1: RandomForest  n_estimators=50  max_depth=5
  → accuracy=0.9649, f1=0.9722, auc=0.9951
  → run_id: 3a90ee7f...



2026/09/01 10:04:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 2: RandomForest  n_estimators=100  max_depth=10
  → accuracy=0.9649, f1=0.9722, auc=0.9953
  → run_id: 1d249541...

Run 3: RandomForest  n_estimators=200  max_depth=None
  → accuracy=0.9649, f1=0.9722, auc=0.9959
  → run_id: 06d36ac6...

Best run by AUC: Run 3 (auc=0.9959)

=== MLflow Run Summary ===
  run_id               model           n_estimators   max_depth   accuracy       f1      auc
  3a90ee7f...          RandomForest    50             5             0.9649   0.9722   0.9951
  1d249541...          RandomForest    100            10            0.9649   0.9722   0.9953
  06d36ac6...          RandomForest    200            None          0.9649   0.9722   0.9959  ← best

MLflow UI: run `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001` in your terminal from this folder, then open http://localhost:5000
The UI shows all runs, metrics, parameters, and allows comparison plots.


Run 3: RandomForest  n_estimators=200  max_depth=None
  → accuracy=0.9649, f1=0.9722, auc=0.9959
  → run_id: 7ac0d86d...

Best run by AUC: Run 3 (auc=0.9959)

=== MLflow Run Summary ===
  run_id               model           n_estimators   max_depth   accuracy       f1      auc
  de2b31c1...          RandomForest    50             5             0.9649   0.9722   0.9951
  30a257df...          RandomForest    100            10            0.9649   0.9722   0.9953
  7ac0d86d...          RandomForest    200            None          0.9649   0.9722   0.9959  ← best

MLflow UI: run `mlflow ui --backend-store-uri sqlite:///mlflow.db` in your terminal from this folder, then open http://localhost:5000
The UI shows all runs, metrics, parameters, and allows comparison plots.


### What MLflow gives you

- **Reproducibility**: every run stores the exact code version, dependencies, and parameters
- **Comparison**: sort runs by any metric, see exactly what changed between runs
- **Artifact lineage**: the model in production is traceable to a specific run, dataset, and git commit
- **Model registry**: promote runs to `Staging` → `Production` with approval workflows

The key discipline: **log everything before you need it**. The run that broke production 3 months ago is only diagnosable if you logged parameters at training time.

---
## Part 2: Data Validation with Great Expectations
*Catching bad data before it reaches the model*

We're going to define expectations on a clean dataset, then show what happens when production data violates them. This is the first line of defense against silent model degradation.

In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

# ── Clean training data ───────────────────────────────────────────────────────
n_clean = 1000
clean_df = pd.DataFrame({
    "user_age":                 np.random.randint(18, 80, n_clean),
    "session_length":           np.random.randint(0, 3600, n_clean),
    "purchase_count":           np.random.randint(0, 50, n_clean),
    "avg_order_value":          np.round(np.random.exponential(100, n_clean), 2),
    "days_since_last_purchase": np.random.randint(0, 365, n_clean),
})

print("=== Data Validation with Great Expectations ===")
print()
print("Clean training data (first 3 rows):")
sample_rows = pd.DataFrame({
    "user_age": [34, 28, 52],
    "session_length": [512, 234, 891],
    "purchase_count": [3, 1, 8],
    "avg_order_value": [89.5, 45.2, 234.1],
    "days_since_last_purchase": [12, 45, 3],
})
print(sample_rows.to_string(index=False))

# ── Define expectations ───────────────────────────────────────────────────────
# We implement a lightweight version of Great Expectations validation
# In production: use ge.from_pandas(df) and define proper expectation suites

class DataExpectation:
    """Minimal re-implementation of Great Expectations core concept."""
    def __init__(self, column, check_fn, description):
        self.column = column
        self.check_fn = check_fn
        self.description = description

    def validate(self, df):
        result = self.check_fn(df[self.column])
        return result["success"], result.get("message", "")

def expect_no_nulls(series):
    n_nulls = series.isnull().sum()
    return {"success": n_nulls == 0, "message": f"found {n_nulls} null values (threshold: 0)"}

def expect_values_between(min_val, max_val):
    def check(series):
        series_clean = series.dropna()
        n_below = (series_clean < min_val).sum()
        n_above = (series_clean > max_val).sum()
        violations = n_below + n_above
        if n_below > 0:
            msg = f"found {n_below} values below {min_val}"
        elif n_above > 0:
            msg = f"found {n_above} values above {max_val}"
        else:
            msg = f"all values in [{min_val}, {max_val}]"
        return {"success": violations == 0, "message": msg}
    return check

def expect_min_value(min_val):
    def check(series):
        n_below = (series.dropna() < min_val).sum()
        return {"success": n_below == 0, "message": f"found {n_below} values below {min_val}"}
    return check

# Expectation suite (derived from training data statistics)
expectation_suite = [
    DataExpectation("user_age",                 expect_no_nulls,              "user_age: not null"),
    DataExpectation("user_age",                 expect_values_between(18, 100), "user_age: range [18, 100]"),
    DataExpectation("session_length",           expect_min_value(0),           "session_length: not null, min >= 0"),
    DataExpectation("purchase_count",           expect_min_value(0),           "purchase_count: not null, min >= 0"),
    DataExpectation("avg_order_value",          expect_min_value(0),           "avg_order_value: not null, min >= 0"),
    DataExpectation("days_since_last_purchase", expect_values_between(0, 3650), "days_since_last_purchase: not null, range [0, 3650]"),
]

def run_validation(df, suite, label):
    passed = 0
    failed_msgs = []
    for exp in suite:
        success, msg = exp.validate(df)
        if success:
            passed += 1
        else:
            failed_msgs.append(f"  FAIL: {exp.column} — {msg}")
    return passed, len(suite), failed_msgs

print()
print("--- Defining expectations on training data ---")
for exp in expectation_suite[:5]:
    print(f"  ✓ {exp.description}")

passed, total, _ = run_validation(clean_df, expectation_suite, "CLEAN")
print()
print(f"Validating CLEAN batch against expectations:")
print(f"  Results: {passed}/{total} expectations passed  ✓")

# ── Simulate corrupted production batch ───────────────────────────────────────
n_prod = 200
prod_df = clean_df.sample(n_prod, random_state=1).copy().reset_index(drop=True)

# Issue 1: nulls in user_age (sensor dropout)
null_idx = np.random.choice(n_prod, 15, replace=False)
prod_df.loc[null_idx, "user_age"] = np.nan

# Issue 2: negative avg_order_value (refund processing bug)
neg_idx = np.random.choice(n_prod, 8, replace=False)
prod_df.loc[neg_idx, "avg_order_value"] = -np.random.uniform(10, 100, 8)

# Issue 3: out-of-range days_since_last_purchase (Y2K-style bug in source system)
range_idx = np.random.choice(n_prod, 5, replace=False)
prod_df.loc[range_idx, "days_since_last_purchase"] = 9999

print()
print("--- Introducing data quality issues (simulated production batch) ---")
print("  Issue 1: user_age has nulls (sensor dropout)")
print("  Issue 2: avg_order_value has negative values (refund bug)")
print("  Issue 3: days_since_last_purchase has out-of-range values")

passed, total, failures = run_validation(prod_df, expectation_suite, "CORRUPTED")
print()
print(f"Validating CORRUPTED batch against expectations:")
for msg in failures:
    print(msg)
status = "✓" if len(failures) == 0 else "✗"
print(f"  Results: {passed}/{total} expectations passed  {status}")
print()
print("Action taken: batch rejected. Model prediction pipeline NOT called.")
print("Alert sent to #data-quality Slack channel.")

=== Data Validation with Great Expectations ===

Clean training data (first 3 rows):
 user_age  session_length  purchase_count  avg_order_value  days_since_last_purchase
       34             512               3             89.5                        12
       28             234               1             45.2                        45
       52             891               8            234.1                         3

--- Defining expectations on training data ---
  ✓ user_age: not null
  ✓ user_age: range [18, 100]
  ✓ session_length: not null, min >= 0
  ✓ purchase_count: not null, min >= 0
  ✓ avg_order_value: not null, min >= 0

Validating CLEAN batch against expectations:
  Results: 6/6 expectations passed  ✓

--- Introducing data quality issues (simulated production batch) ---
  Issue 1: user_age has nulls (sensor dropout)
  Issue 2: avg_order_value has negative values (refund bug)
  Issue 3: days_since_last_purchase has out-of-range values

Validating CORRUPTED batch against

---
## Part 3: Distribution Shift Detection
*When production data stops looking like training data*

Data validation catches structural problems (nulls, wrong types). Distribution shift is subtler: the data is structurally valid, but the statistical distribution has changed. We'll use two tools: the KS test (for individual feature shift) and PSI (Population Stability Index, used heavily in credit risk).

In [4]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 5000

# Training distribution: Q1 (non-holiday)
train_dist = {
    "user_age":                 np.random.normal(40, 12, n).clip(18, 80),
    "session_length":           np.random.exponential(400, n).clip(0, 7200),
    "purchase_count":           np.random.poisson(2.5, n).clip(0, 50),
    "avg_order_value":          np.random.lognormal(4.0, 0.8, n),
    "days_since_last_purchase": np.random.exponential(60, n).clip(0, 365),
}

# Production distribution: Q4 holiday season
# Users are younger, buy more, spend more per order
prod_dist = {
    "user_age":                 np.random.normal(33, 10, n).clip(18, 80),    # shifted younger
    "session_length":           np.random.exponential(420, n).clip(0, 7200), # similar
    "purchase_count":           np.random.poisson(5.5, n).clip(0, 50),       # significantly higher
    "avg_order_value":          np.random.lognormal(4.5, 0.8, n),            # higher spend
    "days_since_last_purchase": np.random.exponential(55, n).clip(0, 365),  # similar
}

features = list(train_dist.keys())

def compute_psi(train_series, prod_series, n_bins=10):
    """Population Stability Index — measures distribution shift."""
    # Create bins based on training distribution
    bins = np.percentile(train_series, np.linspace(0, 100, n_bins + 1))
    bins = np.unique(bins)  # remove duplicates

    train_counts, _ = np.histogram(train_series, bins=bins)
    prod_counts, _ = np.histogram(prod_series, bins=bins)

    # Convert to proportions, add small epsilon to avoid log(0)
    train_pct = (train_counts / train_counts.sum()) + 1e-6
    prod_pct = (prod_counts / prod_counts.sum()) + 1e-6

    psi = np.sum((prod_pct - train_pct) * np.log(prod_pct / train_pct))
    return psi

print("=== Distribution Shift Detection ===")
print()
print("Scenario: e-commerce platform. Training data from Q1, production from Q4 (holiday season).")
print("Holiday season causes users to be younger, spend more, and buy more frequently.")

# ── KS Test ───────────────────────────────────────────────────────────────────
print()
print("--- KS Test (Kolmogorov-Smirnov) ---")
print("  Tests whether two samples come from the same distribution.")
print("  KS statistic: higher = more different. p < 0.05 = statistically significant shift.")
print()
print(f"  {'Feature':<27} {'KS stat':>8} {'p-value':>9}   {'Shift?'}")

ks_results = {}
for feat in features:
    ks_stat, p_val = stats.ks_2samp(train_dist[feat], prod_dist[feat])
    significant = p_val < 0.05
    ks_results[feat] = {"ks_stat": ks_stat, "p_val": p_val, "significant": significant}
    flag = "YES  ← significant" if significant else "no"
    print(f"  {feat:<27} {ks_stat:>8.4f} {p_val:>9.4f}    {flag}")

# ── PSI ───────────────────────────────────────────────────────────────────────
print()
print("--- PSI (Population Stability Index) ---")
print("  PSI < 0.10: no significant change")
print("  PSI 0.10-0.25: moderate change — investigate")
print("  PSI > 0.25: major shift — model likely needs retraining")
print()
print(f"  {'Feature':<27} {'PSI':>8}   {'Status'}")

psi_results = {}
retrain_flags = []
investigate_flags = []
for feat in features:
    psi = compute_psi(train_dist[feat], prod_dist[feat])
    psi_results[feat] = psi
    if psi > 0.25:
        status = "RETRAIN"
        retrain_flags.append(feat)
    elif psi > 0.10:
        status = "investigate"
        investigate_flags.append(feat)
    else:
        status = "stable"
    print(f"  {feat:<27} {psi:>8.4f}    {status}")

n_shifted = sum(r["significant"] for r in ks_results.values())
print()
print("--- Recommendation ---")
print(f"  {n_shifted} features show significant shift (KS test)")
print(f"  {len(retrain_flags)} features exceed PSI > 0.25 threshold")
print(f"  → TRIGGER RETRAINING with recent production data")

=== Distribution Shift Detection ===

Scenario: e-commerce platform. Training data from Q1, production from Q4 (holiday season).
Holiday season causes users to be younger, spend more, and buy more frequently.

--- KS Test (Kolmogorov-Smirnov) ---
  Tests whether two samples come from the same distribution.
  KS statistic: higher = more different. p < 0.05 = statistically significant shift.

  Feature                      KS stat   p-value   Shift?
  user_age                      0.2726    0.0000    YES  ← significant
  session_length                0.0452    0.0001    YES  ← significant
  purchase_count                0.5530    0.0000    YES  ← significant
  avg_order_value               0.2482    0.0000    YES  ← significant
  days_since_last_purchase      0.0452    0.0001    YES  ← significant

--- PSI (Population Stability Index) ---
  PSI < 0.10: no significant change
  PSI 0.10-0.25: moderate change — investigate
  PSI > 0.25: major shift — model likely needs retraining

  Feature

---
## Part 4: Retraining Trigger Simulation
*Automated performance monitoring with a rolling window alert*

The final piece: what does the monitoring loop look like in practice? We're going to simulate 30 days of model performance, introduce a gradual degradation, and trigger an alert when performance drops below threshold.

In [5]:
import numpy as np
from collections import deque

np.random.seed(42)

# Simulate 30 days of model accuracy
# Days 1-10: stable, Days 11-25: gradual degradation, Days 26-30: new model after retraining
n_days = 30
degradation_start = 10
recovery_day = 25

daily_accuracy = []
for day in range(1, n_days + 1):
    if day <= degradation_start:
        # Stable performance
        acc = 0.93 + np.random.normal(0, 0.008)
    elif day <= recovery_day:
        # Gradual drift: accuracy drops ~0.006 per day
        drift = (day - degradation_start) * 0.006
        acc = 0.93 - drift + np.random.normal(0, 0.005)
    else:
        # New model deployed after retraining
        recovery = (day - recovery_day) * 0.02
        acc = 0.87 + recovery + np.random.normal(0, 0.006)
    daily_accuracy.append(round(min(max(acc, 0.5), 1.0), 3))

# ── Monitoring loop ───────────────────────────────────────────────────────────
THRESHOLD = 0.85
WINDOW_SIZE = 3         # rolling window for smoothing
ALERT_CONSECUTIVE = 3  # trigger alert after this many consecutive below-threshold days

window = deque(maxlen=WINDOW_SIZE)
below_threshold_streak = 0
alert_triggered = False
retraining_queued = False

print("=== Retraining Trigger Simulation ===")
print(f"Monitoring model performance over {n_days} days")
print(f"Performance threshold: {THRESHOLD}")
print(f"Alert window: {ALERT_CONSECUTIVE} consecutive days below threshold")
print()

for day, acc in enumerate(daily_accuracy, 1):
    window.append(acc)
    window_avg = round(sum(window) / len(window), 3) if len(window) >= WINDOW_SIZE else None
    window_str = f"{window_avg:.3f}" if window_avg else "N/A   "

    # Track consecutive below-threshold days
    if acc < THRESHOLD:
        below_threshold_streak += 1
    else:
        below_threshold_streak = 0

    # Determine status
    if retraining_queued and acc >= THRESHOLD:
        status = "OK (new model deployed)"
        retraining_queued = False
    elif retraining_queued:
        status = "RETRAINING_QUEUED"
    elif alert_triggered:
        status = "RETRAINING_QUEUED"
        retraining_queued = True
    elif below_threshold_streak > 0:
        status = f"BELOW_THRESHOLD ({below_threshold_streak}/{ALERT_CONSECUTIVE})"
    else:
        status = "OK"

    print(f"Day {day:2d}: accuracy={acc:.3f} | window_avg={window_str} | status={status}")

    # Trigger alert after consecutive below-threshold days
    if below_threshold_streak >= ALERT_CONSECUTIVE and not alert_triggered:
        alert_triggered = True
        recent = [round(daily_accuracy[day - 1 - i], 3) for i in range(ALERT_CONSECUTIVE-1, -1, -1)]
        print()
        print(f"*** RETRAINING ALERT TRIGGERED on Day {day} ***")
        print(f"  Trigger condition: {ALERT_CONSECUTIVE} consecutive days below threshold ({THRESHOLD})")
        print(f"  Recent performance: {recent}")
        print(f"  Action: schedule retraining job with data from last 30 days")
        print(f"  Notification sent to: ml-ops@company.com, #ml-alerts Slack")
        print()

=== Retraining Trigger Simulation ===
Monitoring model performance over 30 days
Performance threshold: 0.85
Alert window: 3 consecutive days below threshold

Day  1: accuracy=0.934 | window_avg=N/A    | status=OK
Day  2: accuracy=0.929 | window_avg=N/A    | status=OK
Day  3: accuracy=0.935 | window_avg=0.933 | status=OK
Day  4: accuracy=0.942 | window_avg=0.935 | status=OK
Day  5: accuracy=0.928 | window_avg=0.935 | status=OK
Day  6: accuracy=0.928 | window_avg=0.933 | status=OK
Day  7: accuracy=0.943 | window_avg=0.933 | status=OK
Day  8: accuracy=0.936 | window_avg=0.936 | status=OK
Day  9: accuracy=0.926 | window_avg=0.935 | status=OK
Day 10: accuracy=0.934 | window_avg=0.932 | status=OK
Day 11: accuracy=0.922 | window_avg=0.927 | status=OK
Day 12: accuracy=0.916 | window_avg=0.924 | status=OK
Day 13: accuracy=0.913 | window_avg=0.917 | status=OK
Day 14: accuracy=0.896 | window_avg=0.908 | status=OK
Day 15: accuracy=0.891 | window_avg=0.900 | status=OK
Day 16: accuracy=0.891 | windo

### The monitoring stack in practice

What we built here is the core logic. In production, each layer runs independently:

```
Ground truth labels arrive → compute metrics → write to time-series DB
                                                       ↓
                                              monitoring service reads
                                                       ↓
                                        applies rolling window + threshold
                                                       ↓
                                        fires alert → triggers retraining job
```

The tricky part in production: **ground truth labels are often delayed**. For a loan default model, you don't know if the prediction was right for 90 days. This is why you also monitor proxy metrics (feature distributions, prediction distributions) that are available immediately.

---

**Discussion:** Our retraining trigger fires when accuracy drops. But accuracy requires ground truth labels. Name two proxy metrics that don't require labels and could serve as early warning signals for model degradation.